# XAI-SplitShield — Analysis Notebook

This notebook provides interactive analysis of:
1. Attribution heatmaps (SHAP / LRP) on clean vs. poisoned smashed data
2. AAS score distributions over training
3. AWSA trust weight evolution (multi-client)
4. ASR / CA comparison across attacks and defenses
5. Ablation study visualization

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 130

from data.dataset_loader import get_dataset
from data.poisoning import get_attack, PoisonedDataLoader
from models.resnet import build_split_resnet18, get_smashed_shape
from defense.xai_splitshield import XAISplitShield
from utils.metrics import compute_asr, compute_clean_accuracy
from utils.visualization import (
    plot_attribution_heatmap, plot_aas_timeseries,
    plot_awsa_trust_weights, plot_asr_ca_comparison, plot_training_curves
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Load Dataset and Models

In [ ]:
DATASET     = 'cifar10'
SPLIT_LAYER = 2
IMAGE_SIZE  = 32
BATCH_SIZE  = 64

train_loader, test_loader, num_classes = get_dataset(
    name=DATASET, root='../data/raw',
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)
print(f'Dataset: {DATASET}  |  Classes: {num_classes}')

client_model, server_model = build_split_resnet18(
    num_classes=num_classes, split_layer=SPLIT_LAYER
)
client_model = client_model.to(DEVICE)
server_model = server_model.to(DEVICE)

smashed_shape = get_smashed_shape(client_model, IMAGE_SIZE, device=DEVICE)
print(f'Smashed data shape: {smashed_shape}  (d={int(np.prod(smashed_shape))})')

## 2. Instantiate Attack and Defense

In [ ]:
ATTACK_NAME = 'wanet'   # badnets | blended | wanet | lira

cfg = {
    'target_class': 0, 'trigger_size': 3, 'trigger_alpha': 0.1,
    'sda_alpha': 0.6, 'sda_n_samples': 20,
    'aas_lambda': 0.5, 'aas_ema_decay': 0.95, 'aas_fpr': 0.01,
    'agns_epsilon_acc': 0.01, 'agns_p_mask': 1.0,
    'awsa_beta': 2.0, 'awsa_window': 10,
}

attack = get_attack(
    name=ATTACK_NAME, cfg=cfg,
    image_size=IMAGE_SIZE, num_classes=num_classes, device=DEVICE
)

shield = XAISplitShield(
    server_model=server_model,
    smashed_shape=tuple(smashed_shape),
    device=DEVICE,
    cfg=cfg,
    num_clients=1,
)
print(f'Attack: {ATTACK_NAME}  |  Defense: XAI-SplitShield')

## 3. Warm-up and Calibration

In [ ]:
WARMUP_BATCHES = 40

client_model.eval()
for i, (images, labels) in enumerate(train_loader):
    if i >= WARMUP_BATCHES:
        break
    images = images.to(DEVICE)
    labels = labels.to(DEVICE)
    with torch.no_grad():
        z = client_model(images)
    shield.warmup(z, labels)

shield.calibrate()
print(f'Calibration done. Threshold τ = {shield.aas_detector.threshold:.4f}')

## 4. Attribution Heatmaps: Clean vs. Poisoned

In [ ]:
# Get one clean and one poisoned batch
images, labels = next(iter(test_loader))
images = images.to(DEVICE)

poisoned_images, _ = attack.inject_test(images[:4].clone())

with torch.no_grad():
    z_clean   = client_model(images[:4])
    z_poisoned = client_model(poisoned_images)

phi_clean   = shield.sda.attribute_magnitude(z_clean)
phi_poisoned = shield.sda.attribute_magnitude(z_poisoned)

fig1 = plot_attribution_heatmap(
    image=images[0].cpu(),
    phi=phi_clean[0].cpu(),
    title='Attribution Map — Clean Input',
)
plt.show()

fig2 = plot_attribution_heatmap(
    image=poisoned_images[0].cpu().detach(),
    phi=phi_poisoned[0].cpu(),
    title=f'Attribution Map — Poisoned Input ({ATTACK_NAME})',
)
plt.show()

## 5. AAS Score Distribution: Clean vs. Poisoned

In [ ]:
aas_scores_clean = []
aas_scores_poisoned = []
N_BATCHES = 30

client_model.eval()
for i, (images, labels) in enumerate(test_loader):
    if i >= N_BATCHES:
        break
    images = images.to(DEVICE)
    with torch.no_grad():
        z_c = client_model(images)
    score_c = shield.aas_detector.compute(z_c, update_baseline=False)
    aas_scores_clean.append(score_c)

    poisoned, _ = attack.inject_test(images)
    with torch.no_grad():
        z_p = client_model(poisoned)
    score_p = shield.aas_detector.compute(z_p, update_baseline=False)
    aas_scores_poisoned.append(score_p)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(aas_scores_clean,   bins=20, alpha=0.7, color='#2196F3', label='Clean')
ax.hist(aas_scores_poisoned, bins=20, alpha=0.7, color='#F44336', label='Poisoned')
ax.axvline(shield.aas_detector.threshold, color='green', linestyle='--',
           linewidth=2, label=f'Threshold τ={shield.aas_detector.threshold:.3f}')
ax.set_xlabel('AAS Score'); ax.set_ylabel('Count')
ax.set_title('AAS Score Distributions: Clean vs. Poisoned')
ax.legend(); plt.tight_layout(); plt.show()

print(f'Clean   AAS: mean={np.mean(aas_scores_clean):.4f}  std={np.std(aas_scores_clean):.4f}')
print(f'Poisoned AAS: mean={np.mean(aas_scores_poisoned):.4f}  std={np.std(aas_scores_poisoned):.4f}')

## 6. AGNS — Neuron Suppression Visualization

In [ ]:
images_test, labels_test = next(iter(test_loader))
poisoned_test, _ = attack.inject_test(images_test[:8].to(DEVICE))

with torch.no_grad():
    z_p = client_model(poisoned_test)

z_defended, report = shield.defend(z_p, labels_test[:8].to(DEVICE))

print(f'AAS Score:       {report.aas_score:.4f}')
print(f'Is Poisoned:     {report.is_poisoned}')
print(f'Neurons Suppressed: {report.num_suppressed}')
print(f'Detection Threshold: {report.detection_threshold:.4f}')

if report.suppression_mask is not None:
    mask_np = report.suppression_mask[0].mean(dim=0).cpu().numpy()
    plt.figure(figsize=(5, 4))
    plt.imshow(mask_np, cmap='RdYlGn', vmin=0, vmax=1)
    plt.colorbar()
    plt.title(f'AGNS Suppression Mask ({report.num_suppressed} neurons zeroed)')
    plt.tight_layout(); plt.show()

## 7. ASR / CA Results Summary

In [ ]:
# Example: Load results from a completed run_all.sh
import json, glob
from pathlib import Path

LOG_DIR = '../results'
results = {}

for summary_path in Path(LOG_DIR).rglob('summary.json'):
    with open(summary_path) as f:
        data = json.load(f)
    name = data.get('run_name', summary_path.parent.name)
    results[name] = {'asr': data.get('final_asr', 0) / 100,
                     'ca':  data.get('final_ca', 0) / 100}

if results:
    fig = plot_asr_ca_comparison(results, metric='asr',
                                  title='ASR Comparison Across Experiments')
    plt.show()
else:
    print(f'No results found in {LOG_DIR}. Run experiments first.')